# JupyterHub environment check

Confirms that a freshly spawned server on this hub actually works: the kernel runs, `%pip` works,
the home directory is writable, the memory envelope is what the profile promised, and both the
public internet and the in-cluster Ceph gateway are reachable.

**Run All Cells.** The last line is either `JupyterHub is ready` or a list of what failed. Run it
once per spawner profile -- the memory envelope is what differs between them.

This checks the *hub*. To confirm a specific assignment's datasets are ready, run that assignment's
`00-environment-check.ipynb`, in its answer-key repo. Failures are explained in
[docs/5_verify_hub.md](../docs/5_verify_hub.md).

In [ ]:
name = "YOUR NAME HERE"
date = "MM/DD/YYYY"

In [ ]:
# The union of image/requirements.txt -- every assignment's dependencies, which is
# what the one combined nids-hub image carries.
#
# On the nids-hub image every line is already satisfied and this is a no-op; that
# no-op is part of what is being verified. On the hosted NRP hub this really does
# install into your home directory and can take several minutes (pyspark is large).
%pip install dpkt pandas pyarrow matplotlib geoip2 maxminddb py-radix pelicanfs pybgpkit-parser pyspark tldextract dnspython numpy requests neo4j python-dotenv

In [ ]:
# --- check runner -------------------------------------------------------------
# Self-contained on purpose: this notebook is handed around on its own, so the
# runner is duplicated here rather than imported from a shared module.
import os
import socket
import sys
import urllib.error
import urllib.request

RESULTS = []  # (label, "ok" | "fail" | "warn")


class check:
    """`with check("label") as c:` -- runs the body, records the outcome, never raises.

    Set `c.note = "..."` inside the body to add detail to the printed line.
    `required=False` downgrades a failure to a warning, which does not block "ready".
    """

    def __init__(self, label, required=True):
        self.label = label
        self.required = required
        self.note = ""

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        if exc is None:
            RESULTS.append((self.label, "ok"))
            print(f"[ ok ] {self.label}" + (f"  ({self.note})" if self.note else ""))
        else:
            RESULTS.append((self.label, "fail" if self.required else "warn"))
            tag = "FAIL" if self.required else "warn"
            print(f"[{tag}] {self.label}  ->  {exc.__class__.__name__}: {exc}")
        return True  # swallow the exception so the remaining checks still run


def http_head(url, timeout=60):
    """HEAD a URL. Raises on a non-2xx status or a connection failure."""
    req = urllib.request.Request(url, method="HEAD")
    with urllib.request.urlopen(req, timeout=timeout) as response:
        return response.status, response.headers


def http_first_bytes(url, n=64, timeout=60):
    """Range-GET the first n bytes of a URL -- never pulls the whole object."""
    req = urllib.request.Request(url, headers={"Range": f"bytes=0-{n - 1}"})
    with urllib.request.urlopen(req, timeout=timeout) as response:
        return response.read(n)


def human_size(headers):
    size = headers.get("Content-Length")
    return f"{int(size) / 2**20:.0f} MiB" if size else "size unknown"


def mem_limit_gib():
    """This pod's memory limit in GiB (cgroup v2, then v1). None if unlimited."""
    try:
        raw = open("/sys/fs/cgroup/memory.max").read().strip()
        if raw != "max":
            return int(raw) / 2**30
    except OSError:
        pass
    try:
        raw = int(open("/sys/fs/cgroup/memory/memory.limit_in_bytes").read().strip())
        if raw < 2**60:
            return raw / 2**30
    except OSError:
        pass
    return None


def report(subject=""):
    """Print the verdict. Call this last."""
    failed = [label for label, state in RESULTS if state == "fail"]
    warned = [label for label, state in RESULTS if state == "warn"]
    tail = f" for {subject}" if subject else ""
    print()
    for label in warned:
        print(f"warning: {label} did not pass -- not required, see the note above")
    if failed:
        print(f"NOT ready{tail} - {len(failed)} of {len(RESULTS)} checks failed:")
        for label in failed:
            print(f"  - {label}")
        print("\nWhat each failure means: nids-setup/docs/5_verify_hub.md")
    else:
        print(f"JupyterHub is ready{tail}")

In [ ]:
# --- in-cluster Ceph (NRP-internal hostname) -----------------------------------
CEPH = "http://rook-ceph-rgw-nautiluss3.rook"


def _ceph(call, path, *args, **kwargs):
    try:
        return call(f"{CEPH}/{path}", *args, **kwargs)
    except urllib.error.URLError as exc:
        if isinstance(getattr(exc, "reason", None), socket.gaierror):
            raise RuntimeError(
                f"{CEPH} did not resolve. That hostname only exists inside the NRP "
                "cluster -- this will never work from a laptop."
            ) from None
        raise


def ceph_head(path, **kwargs):
    return _ceph(http_head, path, **kwargs)


def ceph_first_bytes(path, n=64, **kwargs):
    return _ceph(http_first_bytes, path, n, **kwargs)

In [ ]:
print("--- shared: what every NIDS assignment needs ---\n")

with check("python and kernel") as c:
    assert sys.version_info >= (3, 9), f"python {sys.version.split()[0]} is older than 3.9"
    c.note = f"python {sys.version.split()[0]}"

with check("home directory is writable") as c:
    probe = os.path.expanduser("~/.nids-hub-check")
    with open(probe, "w") as fh:
        fh.write("ok")
    os.remove(probe)
    stat = os.statvfs(os.path.expanduser("~"))
    c.note = f"{stat.f_bavail * stat.f_frsize / 2**30:.1f} GiB free"

with check("memory and cpu") as c:
    gib = mem_limit_gib()
    c.note = (f"{gib:.1f} GiB limit" if gib else "no cgroup limit visible") + f", {os.cpu_count()} cpu"

with check("shared imports") as c:
    import matplotlib
    import numpy
    import pandas
    import requests
    c.note = f"pandas {pandas.__version__}, numpy {numpy.__version__}"

with check("external egress (pypi)") as c:
    status, _ = http_head("https://pypi.org/simple/")
    c.note = f"HTTP {status}"

with check("public dns resolution") as c:
    socket.getaddrinfo("object.openintel.nl", 443)

with check("in-cluster ceph gateway") as c:
    # as2org.jsonl: read by 2 of the 6 assignments, and the smallest proof the
    # gateway is both resolvable and serving objects.
    status, headers = ceph_head("caida/as2org/as2org.jsonl")
    c.note = f"HTTP {status}, as2org.jsonl {human_size(headers)}"

In [ ]:
# Spark is a capability of the nids-hub image, not something every assignment uses,
# so these are warnings rather than failures: on the hosted NRP hub Spark is
# legitimately absent, and only the DNS assignment needs it.
print("--- nids-hub image extras (informational on the hosted NRP hub) ---\n")

with check("pre-staged spark s3a jars", required=False) as c:
    import glob
    spark_home = os.environ.get("SPARK_HOME", "")
    jars = sorted(
        os.path.basename(p)
        for p in glob.glob(f"{spark_home}/jars/hadoop-aws-*.jar")
        + glob.glob(f"{spark_home}/jars/bundle-*.jar")
    )
    assert jars, f"no hadoop-aws / awssdk bundle jar under {spark_home!r}/jars (see image/Dockerfile)"
    c.note = ", ".join(jars)

with check("spark session (local[*])", required=False) as c:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.master("local[*]").appName("nids-hub-check").getOrCreate()
    c.note = f"spark {spark.version}"
    spark.stop()

In [ ]:
report()